In [ ]:
import cv2
import numpy as np

def color_detection(frame, color_name, lower_bound, upper_bound, min_box_size=500):
    """
    지정된 HSV 색상 범위를 기반으로 객체를 검출하고 바운딩 박스를 그리는 함수
    """
    # 1. BGR 이미지를 HSV 색 공간으로 변환
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    # 2. 색상 마스크 생성 (Red의 경우 튜플/리스트로 2개 범위가 들어올 수 있음)
    if isinstance(lower_bound, list) and isinstance(upper_bound, list):
        mask1 = cv2.inRange(hsv, lower_bound[0], upper_bound[0])
        mask2 = cv2.inRange(hsv, lower_bound[1], upper_bound[1])
        mask = cv2.bitwise_or(mask1, mask2)
    else:
        mask = cv2.inRange(hsv, lower_bound, upper_bound)
    
    # 3. 모폴로지 연산(열림/닫힘)으로 노이즈 제거
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_DILATE, kernel)
    
    # 4. 윤곽선(Contours) 검출
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # 5. 검출된 윤곽선 중 일정 크기 이상인 객체에 박스 및 텍스트 표시
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > min_box_size:
            x, y, w, h = cv2.boundingRect(cnt)
            
            # 색상별 박스 색상 지정 (BGR)
            box_colors = {
                "Red": (0, 0, 255),
                "Green": (0, 255, 0),
                "Blue": (255, 0, 0)
            }
            color = box_colors.get(color_name, (0, 255, 255))
            
            # 사각형 및 라벨 그리기
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, color_name, (x, y - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)
            
    return frame

# --- HSV 색상 범위 설정 (H: 0~179, S: 0~255, V: 0~255) ---
# 빨간색(Red)은 HSV 색상환에서 0도와 180도 양 끝에 걸쳐있으므로 2개 범위 사용
lower_red = [np.array([0, 100, 100]), np.array([170, 100, 100])]
upper_red = [np.array([10, 255, 255]), np.array([180, 255, 255])]

# 초록색(Green) 범위
lower_green = np.array([35, 80, 80])
upper_green = np.array([85, 255, 255])

# 파란색(Blue) 범위
lower_blue = np.array([95, 100, 100])
upper_blue = np.array([130, 255, 255])

min_box_size = 800  # 최소 인식 면적 (환경에 맞게 조절)

# --- 웹캠 실행 ---
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # 원본 프레임에 R, G, B 순차적으로 박스 그리기
    frame = color_detection(frame, "Red", lower_red, upper_red, min_box_size)
    frame = color_detection(frame, "Green", lower_green, upper_green, min_box_size)
    frame = color_detection(frame, "Blue", lower_blue, upper_blue, min_box_size)
    
    cv2.imshow('Color Detection', frame)
    
    # 'q' 키를 누르면 종료
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()